In [ ]:
from datetime import datetime
import pandas as pd

# BI Inflation Rate

Source: https://www.bi.go.id/id/statistik/indikator/data-inflasi.aspx

In [ ]:
def average_rate(dataframe, start_date, end_date):

    start_date = pd.to_datetime(start_date)
    end_date = pd.to_datetime(end_date)

    if start_date > end_date:
        print("Start date can't be greater than the end date!")
    elif end_date > dataframe['Date'].max():
        end_date = dataframe['Date'].max()

    mask = (dataframe['Date'] >= start_date) & (dataframe['Date'] <= end_date)
    filtered_df = dataframe[mask]

    if filtered_df.empty:
        print(f"No data found between {start_date} and {end_date}.")
        return None

    average_rate = filtered_df['Actual_Rate'].mean()
    return average_rate

In [ ]:
from datetime import datetime
import pandas as pd

def datecov(date_str):

    id_months = {
        "januari": "01", "februari": "02", "maret": "03", "april": "04",
        "mei": "05", "juni": "06", "juli": "07", "agustus": "08",
        "september": "09", "oktober": "10", "november": "11", "desember": "12"
    }

    try:
        # Split the string into month and year, handling any accidental extra spaces
        month_name, year = date_str.strip().split()

        # Look up the month number (lowercased to ensure case-insensitivity)
        month_num = id_months[month_name.lower()]

        # Return the formatted date string with "01" as the default day
        return f"01/{month_num}/{year}"

    except (ValueError, KeyError):
        raise ValueError(f"Invalid date format or month name: '{date_str}'")
# ABOVE WRITTEN WITH GEMINI

df = pd.read_excel('/Data Inflasi.xlsx', header=None)

df = df.iloc[4:,1:3]

df.columns = ["Date", "Rate"]
df = df[1:]

df['Date'] = df["Date"].apply(datecov)
df['Date'] = pd.to_datetime(df["Date"], format='%d/%m/%Y')

df['Actual_Rate'] = df['Rate'].str.replace('%', '', regex=False).astype(float) / 100


#last 3 years and 5 years
print(average_rate(df, '2023-08-01', '2026-05-01'))
print(average_rate(df, '2021-08-01', '2026-05-01'))

# 2024
print(average_rate(df, '2021-08-01', '2024-07-01')) # last 3 years
print(average_rate(df,'2019-08-01', '2024-07-01')) # last 5 years

# 2025
print(average_rate(df,'2022-08-01', '2025-07-01')) # last 3 years
print(average_rate(df,'2020-08-01', '2025-07-01')) # last 5 years

# 2026
print(average_rate(df,'2023-08-01', '2026-07-01')) # last 3 years
print(average_rate(df,'2021-08-01', '2026-07-01')) # last 5 years

0.02395882352941177
0.029491379310344824
0.03389444444444444
0.02873833333333333
0.03017777777777778
0.026333333333333334
0.02395882352941177
0.029491379310344824


# Fed Inflation Rate
Source: https://inflationdata.com/articles/current-inflation/#Current_Inflation_Table

In [ ]:
fed_df = pd.read_excel("/Annual Inflation Table.xlsx")


In [ ]:
# Melt the DataFrame to transform monthly columns into rows
# Exclude 'Ave.' column from melting as it's not a month
month_columns = [col for col in fed_df.columns if col not in ['Year', 'Ave.']]
fed_df_melted = fed_df.melt(id_vars=['Year'], value_vars=month_columns, var_name='Month', value_name='Rate')

# Drop rows where 'Rate' is NaN (e.g., future months or missing data)
fed_df_melted = fed_df_melted.dropna(subset=['Rate'])

# Convert month names to numbers for date parsing
month_to_num = {
    'Jan': '01', 'Feb': '02', 'Mar': '03', 'Apr': '04', 'May': '05', 'Jun': '06',
    'Jul': '07', 'Aug': '08', 'Sep': '09', 'Oct': '10', 'Nov': '11', 'Dec': '12'
}
fed_df_melted['Month_num'] = fed_df_melted['Month'].map(month_to_num)

# Create a 'Date' column
fed_df_melted['Date'] = fed_df_melted['Year'].astype(str) + '-' + fed_df_melted['Month_num'] + '-01'
fed_df_melted['Date'] = pd.to_datetime(fed_df_melted['Date'])

# Convert 'Rate' to float and then calculate 'Actual_Rate'
fed_df_melted['Rate'] = pd.to_numeric(fed_df_melted['Rate'])
fed_df_melted['Actual_Rate'] = fed_df_melted['Rate'] / 100

# Select and reorder columns to match the 'BI Inflation Rate' format
fed_df = fed_df_melted[['Date', 'Rate', 'Actual_Rate']]



In [ ]:
print(average_rate(fed_df, '2023-08-01', '2026-05-01'))
print(average_rate(fed_df, '2021-08-01', '2026-05-01'))

0.029711764705882354
0.04489137931034482


# Inflation Expectations
Source: https://www.clevelandfed.org/indicators-and-data/inflation-expectations

In [ ]:
ie_df1 = pd.read_excel('/Inflation expectations.xlsx', sheet_name='Expected Inflation')
ie_df2 = pd.read_excel('/Inflation expectations.xlsx', sheet_name='Ten-year Expected Chart')
ie_df3 = pd.read_excel('/Inflation expectations.xlsx', sheet_name='Real Interest Rate')

ie_df1['Model Output Date'] = pd.to_datetime(ie_df1['Model Output Date'])
ie_df2['Model Output Date'] = pd.to_datetime(ie_df1['Model Output Date'])
ie_df3['Model Output Date'] = pd.to_datetime(ie_df1['Model Output Date'])

ie_df1.rename(columns={'Model Output Date': 'Date'}, inplace=True)
ie_df2.rename(columns={'Model Output Date': 'Date'}, inplace=True)
ie_df3.rename(columns={'Model Output Date': 'Date'}, inplace=True)

# Prepare renaming dictionary for inflation expectation columns
rename_mapping = {}
for i in range(1, 31):
    old_name = f' {i} year Expected Inflation'
    new_name = f'{i}Y'
    if old_name in ie_df1.columns:
        rename_mapping[old_name] = new_name

# Apply renaming to ie_df1
ie_df1.rename(columns=rename_mapping, inplace=True)

## Minimum and Maximum (last 12 months)

In [ ]:
# Find the latest date in ie_df1
latest_date = ie_df1['Date'].max()

# Calculate the date 12 months prior to the latest date
twelve_months_ago = latest_date - pd.DateOffset(months=12)

# Filter ie_df1 for the last 12 months
ie_df1_last_12_months = ie_df1[ie_df1['Date'] >= twelve_months_ago]

min_list = []
max_list = []
head_col = []

for col in ie_df1_last_12_months.columns:
    if col != 'Date':
        min_list.append(ie_df1_last_12_months[col].min())
        max_list.append(ie_df1_last_12_months[col].max())
        head_col.append(col)

minmax_df = pd.DataFrame([min_list,max_list], columns=head_col)

# the first row = min
# second row = max

## Minimum and Maximum (latest year - latest year+1)

In [ ]:
print(min(minmax_df.iloc[0,0], minmax_df.iloc[0,1]))
print(max(minmax_df.iloc[0,0], minmax_df.iloc[0,1]))

0.0225765620574154
0.02295295211914522


## Minimum and Maximum (>latest year + 1)

In [ ]:
print(min(minmax_df.iloc[0, 1:]))
print(max(minmax_df.iloc[0, 1:]))

0.0217748608068885
0.0237009303061479


## Minimum and Maximum (latest year - 1)

In [ ]:
ie_df1_last_1_year = ie_df1[ie_df1['Date'].dt.year == latest_date.year - 1]

In [ ]:
print(min(ie_df1_last_1_year['1Y']))
print(max(ie_df1_last_1_year['1Y']))

0.02183852723124486
0.03192868843164016
